# FreeFine Final Geometry — Combined Metric Resume on Account A (T4×2)

Use this **instead of rerunning the three separate metric notebooks**.

**v2 fix:** serially preloads the official OpenAI CLIP ViT-B/32 checkpoint before the two GPU workers start, and applies a targeted PyTorch 2.6 compatibility fix to OpenAI CLIP's trusted checkpoint fallback loader. This avoids the concurrent first-download/cache race observed in v1.

It resumes the three existing partial evaluations and computes only the unique missing work.

### Required inputs
Attach:
1. the **4 completed generation shard notebook outputs**,
2. the **3 previous partial metric notebook outputs** (`05`, `06`, `07`),
3. `freefine-sample-metadata`,
4. `geobench2d-coarse-img`,
5. `geobench2d-metrics-subset`.

Total: **10 inputs**.

### Why this is much cheaper
The completed `all_5677`, `resize_2635`, and `move_1439` results are reused from their saved JSONs.

Among the missing groups:
- `rotate_1603` is byte-identical in all 3 final models → evaluate once.
- `resize_nonsevere_1876` is byte-identical in the two SGR models → evaluate once for both.
- `resize_severe_759` differs across baseline / EPSREC / MIDHF+EPSREC → evaluate all 3.

So the remaining work is **6 unique metric jobs instead of 9**.

Run with **T4×2, Internet ON**. The notebook verifies the shared-image identity before reusing any metric result.


### If v1 already ran partially
Save the current v1 version and attach its notebook output as an **additional input** to this v2 notebook.
v2 automatically selects the most complete `final_metric_results_*.json` available, so completed work is preserved.


In [1]:

# ===== 1. Clone exact FreeFine source + clock =====
import os,time,subprocess,glob,json,shutil,csv,hashlib,math,socket,re
from collections import defaultdict,Counter

NB_START=time.time()
COMMIT="4c9fdb971572b32edbeac13464659274c28decbb"
subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True,check=True
)
print("✓ cloned + pinned FreeFine",COMMIT[:14])


✓ cloned + pinned FreeFine 4c9fdb971572b3


In [2]:

%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 73.9 MB/s eta 0:00:00
✓ metric_env OK


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.36s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 446ms
 Downloaded torchaudio
 Downloaded networkx
 Downloaded pillow
 Downloaded sympy
 Downloaded torchvision
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded numpy
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-curand-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded triton
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 29.96s
Installed 27 packages in 427ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1

In [3]:

# ===== Deterministic official metric patches =====
import pathlib,re,py_compile,os
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"
s=mp.read_text()
s=s.replace("args.3d","getattr(args,'3d')")
anchor="    args = parser.parse_args()\n"
assert anchor in s
seed_patch=anchor+"""    import random as _random, numpy as _np
    try:
        import torch as _torch
    except Exception:
        _torch=None
    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))
    _random.seed(_metric_seed); _np.random.seed(_metric_seed)
    if _torch is not None:
        _torch.manual_seed(_metric_seed)
        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)
"""
s=s.replace(anchor,seed_patch,1)
mp.write_text(s)

for f in [mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace(
        "stabilityai/stable-diffusion-2-1",
        "sd2-community/stable-diffusion-2-1"
    ))
md=mr/"MD"/"mean_distance.py"
s=md.read_text()
needle="all_dist = []"
assert needle in s
s=s.replace(
    needle,
    needle+"\n    import torch as _st, os as _os; "
           "_seed=int(_os.environ.get('FF_MD_SEED','42')); "
           "_st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)",
    1
)
md.write_text(s)
for f in [mr/"main.py",mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    py_compile.compile(str(f),doraise=True)
print("✓ deterministic metric patches installed")

# ===== PyTorch 2.6 / OpenAI CLIP compatibility =====
# The pinned OpenAI CLIP loader's fallback torch.load call does not specify
# weights_only. PyTorch 2.6 changed the default to True. For this exact,
# trusted official CLIP checkpoint loader, restore the historical behavior.
import glob as _glob, pathlib as _pathlib
_clip_py=_glob.glob("/kaggle/temp/metric_env/lib/python3.10/site-packages/clip/clip.py")
assert len(_clip_py)==1,_clip_py
_cp=_pathlib.Path(_clip_py[0])
_cs=_cp.read_text()
_old='state_dict = torch.load(opened_file, map_location="cpu")'
_new='state_dict = torch.load(opened_file, map_location="cpu", weights_only=False)'
if _old in _cs:
    _cs=_cs.replace(_old,_new,1)
    _cp.write_text(_cs)
elif _new not in _cs:
    raise AssertionError("Unexpected OpenAI CLIP loader source; refusing broad torch.load patch")
print("✓ OpenAI CLIP PyTorch-2.6 compatibility patch installed")


✓ deterministic metric patches installed
✓ OpenAI CLIP PyTorch-2.6 compatibility patch installed


In [4]:

# ===== Combined resume: assemble all 3 final models + load partial metric JSONs =====
import os,glob,json,csv,shutil,math,hashlib,re,filecmp
from collections import defaultdict,Counter

EXPECTED_SHA="17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802"
MODELS={
    "FREEFINE_BASELINE":"baseline",
    "SGR_EPSREC":"SGR_EPSREC",
    "SGR_MIDHF_EPSREC":"SGR_MIDHF_EPSREC",
}

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs:
        raise FileNotFoundError(f"Missing {desc}: {pattern}")
    return sorted(xs,key=lambda x:(len(x),x))[0]

# Dataset inputs (same as the original metric notebooks)
CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
           if os.path.isdir(f"{c}/source_img"))
COARSE=one("/kaggle/input/**/coarse_img/*/*/*.png","coarse_img").split("/coarse_img/")[0]+"/coarse_img"
ANNP=one("/kaggle/input/**/annotation_2d.json","annotation_2d.json")
META=one("/kaggle/input/**/sample_metadata.csv","sample_metadata.csv")

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}",d)

d=f"{GEO}/Geo-Bench-2D/coarse_img"
if os.path.lexists(d):
    os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
os.symlink(COARSE,d)
shutil.copy(ANNP,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))

meta=[]
for r in csv.DictReader(open(META)):
    if r["edit_type"] not in ("move","rotate","resize"):
        continue
    d,i,e=str(r["da_n"]),str(r["ins_id"]),str(r["case_id"])
    ep=ann[d]["instances"][i][e]["edit_param"]
    sx,sy=float(ep[6]),float(ep[7])
    scale=math.sqrt(sx*sy)
    rr=dict(r)
    rr.update({
        "da_n":d,"ins_id":i,"case_id":e,"scale_exact":scale,
        "affine_severe":(r["edit_type"]=="resize" and (scale>=1.5 or scale<=0.6))
    })
    meta.append(rr)

assert len(meta)==5677
assert Counter(r["edit_type"] for r in meta)=={"resize":2635,"rotate":1603,"move":1439}
assert sum(r["affine_severe"] for r in meta)==759

# Exact-affine benchmark fingerprint
lines=[]
for r in sorted(meta,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"])):
    vals=[float(x) for x in ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["edit_param"]]
    lines.append(
        f"{r['da_n']}|{r['ins_id']}|{r['case_id']}|{r['edit_type']}|"+
        "|".join(f"{x:.12g}" for x in vals)
    )
sha=hashlib.sha256("\n".join(lines).encode()).hexdigest()
assert sha==EXPECTED_SHA,(sha,EXPECTED_SHA)
print("✓ exact benchmark fingerprint",sha)

def category(r):
    if r["edit_type"]=="move": return "move"
    if r["edit_type"]=="rotate": return "rotate"
    return "resize_severe" if r["affine_severe"] else "resize_nonsevere"

offset={"move":0,"rotate":1,"resize_nonsevere":0,"resize_severe":2}
bycat=defaultdict(list)
for r in meta:
    bycat[category(r)].append(r)

row_shard={}
for cat,rows in bycat.items():
    rows=sorted(rows,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"]))
    for j,r in enumerate(rows):
        row_shard[(r["da_n"],r["ins_id"],r["case_id"])]=(j+offset[cat])%4

def pc(p):
    return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0

# Find all 4 completed generation shard outputs for every model.
ROOTS={}
for slug,model_dir in MODELS.items():
    ROOTS[slug]={}
    for s in range(4):
        cands=glob.glob(f"/kaggle/input/**/final_geometry_full/shard_{s}/{model_dir}",recursive=True)
        if not cands:
            raise FileNotFoundError(f"Missing generation shard {s} for {slug}")
        root=max(cands,key=pc)
        ROOTS[slug][s]=root
        print(slug,"shard",s,"pngs",pc(root),root)

# Assemble the 3 exact 5,677 image trees as symlinks.
EVALROOTS={}
for slug in MODELS:
    ev=f"{GEO}/gen_eval/{slug}"
    if os.path.exists(ev):
        shutil.rmtree(ev)
    os.makedirs(ev,exist_ok=True)
    missing=[]
    for r in meta:
        k=(r["da_n"],r["ins_id"],r["case_id"])
        s=row_shard[k]
        src=f"{ROOTS[slug][s]}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
        dst=f"{ev}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
        if not os.path.exists(src):
            missing.append((k,s,src))
            continue
        os.makedirs(os.path.dirname(dst),exist_ok=True)
        os.symlink(os.path.realpath(src),dst)
    assert not missing,missing[:20]
    assert len(glob.glob(ev+"/**/*.png",recursive=True))==5677
    EVALROOTS[slug]=ev
    print("✓ assembled",slug,"5677/5677")

def keys(pred):
    return [(r["da_n"],r["ins_id"],r["case_id"]) for r in meta if pred(r)]

GROUPS={
    "all_5677":keys(lambda r:True),
    "move_1439":keys(lambda r:r["edit_type"]=="move"),
    "rotate_1603":keys(lambda r:r["edit_type"]=="rotate"),
    "resize_2635":keys(lambda r:r["edit_type"]=="resize"),
    "resize_severe_759":keys(lambda r:r["edit_type"]=="resize" and r["affine_severe"]),
    "resize_nonsevere_1876":keys(lambda r:r["edit_type"]=="resize" and not r["affine_severe"]),
}
print("groups",{k:len(v) for k,v in GROUPS.items()})

METRICS=["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"]
def complete(v):
    return isinstance(v,dict) and all(k in v for k in METRICS)

# Load the most complete partial JSON for each model from the 3 old metric notebook outputs
# (and, on a rerun, from a prior combined-resume output).
RESULTS={}
for slug in MODELS:
    cands=glob.glob(f"/kaggle/input/**/final_metric_results_{slug}.json",recursive=True)
    if not cands:
        raise FileNotFoundError(
            f"Missing partial metric JSON for {slug}. Attach the corresponding previous metric notebook output."
        )
    scored=[]
    for p in cands:
        try:
            d=json.load(open(p))
            score=sum(complete(d.get(g,{})) for g in GROUPS)
            scored.append((score,os.path.getsize(p),p,d))
        except Exception:
            pass
    assert scored,f"No readable result JSON for {slug}"
    score,_,p,d=max(scored,key=lambda x:(x[0],x[1]))
    RESULTS[slug]=d
    print("resume",slug,"from",p,"complete groups",
          [g for g in GROUPS if complete(d.get(g,{}))])

# We only reuse a group's metric result across models when the generated PNGs are
# provably byte-identical by construction. Verify every file before copying metrics.
def assert_group_identical(slug_a,slug_b,gname):
    print(f"verifying byte identity: {slug_a} == {slug_b} on {gname} ...",flush=True)
    aroot,broot=EVALROOTS[slug_a],EVALROOTS[slug_b]
    for idx,(d,i,e) in enumerate(GROUPS[gname],1):
        pa=f"{aroot}/{d}/{i}/{e}.png"
        pb=f"{broot}/{d}/{i}/{e}.png"
        if not filecmp.cmp(pa,pb,shallow=False):
            raise AssertionError(f"Shared-branch identity failed: {gname} {d}/{i}/{e}")
        if idx%500==0:
            print(" ",idx,"/",len(GROUPS[gname]),flush=True)
    print("✓ byte-identical",gname,slug_a,slug_b)

# Rotate is baseline in both final proposals.
assert_group_identical("FREEFINE_BASELINE","SGR_EPSREC","rotate_1603")
assert_group_identical("FREEFINE_BASELINE","SGR_MIDHF_EPSREC","rotate_1603")

# Non-severe resize uses the exact same RING8 branch in both SGR proposals.
assert_group_identical("SGR_EPSREC","SGR_MIDHF_EPSREC","resize_nonsevere_1876")

print("✓ shared-branch reuse is safe")


✓ exact benchmark fingerprint 17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802
FREEFINE_BASELINE shard 0 pngs 1419 /kaggle/input/notebooks/georgiostzamouranis/01-full5677-generation-shard0of4-t4x2-v1/final_geometry_full/shard_0/baseline
FREEFINE_BASELINE shard 1 pngs 1419 /kaggle/input/notebooks/giorgostzam/01-full5677-generation-shard1of4-t4x2-v1/final_geometry_full/shard_1/baseline
FREEFINE_BASELINE shard 2 pngs 1420 /kaggle/input/notebooks/papadonikolas/01-full5677-generation-shard2of4-t4x2-v1/final_geometry_full/shard_2/baseline
FREEFINE_BASELINE shard 3 pngs 1419 /kaggle/input/notebooks/gtz19800/01-full5677-generation-shard3of4-t4x2-v1/final_geometry_full/shard_3/baseline
SGR_EPSREC shard 0 pngs 1419 /kaggle/input/notebooks/georgiostzamouranis/01-full5677-generation-shard0of4-t4x2-v1/final_geometry_full/shard_0/SGR_EPSREC
SGR_EPSREC shard 1 pngs 1419 /kaggle/input/notebooks/giorgostzam/01-full5677-generation-shard1of4-t4x2-v1/final_geometry_full/shard_1/SGR_EPSREC


In [5]:

# ===== Serial model-cache preflight BEFORE parallel workers =====
# v1 started two equal-size jobs simultaneously; both could reach the first
# OpenAI CLIP load at nearly the same time and race on the same 338 MB cache file.
# Load it once, in one process, then verify that the cached model opens cleanly.
import os,subprocess,glob

PY="/kaggle/temp/metric_env/bin/python"
pre=os.environ.copy()
pre.update({
    "HF_HOME":"/kaggle/temp/hf",
    "TORCH_HOME":"/kaggle/temp/torch",
    "TOKENIZERS_PARALLELISM":"false",
})

code_str = (
    "import clip, torch\n"
    "print('torch', torch.__version__)\n"
    "model, preprocess = clip.load('ViT-B/32', device='cpu', jit=False)\n"
    "print('CLIP_PREWARM_OK', type(model).__name__)\n"
)

r=subprocess.run([PY,"-c",code_str],env=pre,capture_output=True,text=True)
print(r.stdout)
if r.returncode!=0:
    print(r.stderr[-5000:])
    raise RuntimeError("CLIP serial prewarm failed; do not start metric workers")

cache=glob.glob(os.path.expanduser("~/.cache/clip/*"))
print("CLIP cache:",[(os.path.basename(p),os.path.getsize(p)) for p in cache])
assert cache,"CLIP cache missing after successful preload"
print("✓ CLIP cache prewarmed serially; safe to start GPU workers")


torch 2.6.0+cu124
CLIP_PREWARM_OK CLIP

CLIP cache: [('ViT-B-32.pt', 353976522)]
✓ CLIP cache prewarmed serially; safe to start GPU workers


In [6]:

# ===== Minimal unique missing-work scheduler for Account A =====
# We do NOT rerun all_5677 / resize_2635 / move_1439.
# Shared final branches are measured once:
#   rotate: one metric run for all three models
#   resize_nonsevere: one SGR run for both SGR models
# Unique remaining work = 6 metric jobs, not 9.
import os,json,re,subprocess,time,threading,queue,glob,shutil,pandas as pd,copy

MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
LOGDIR="/kaggle/working/combined_resume_logs"
os.makedirs(LOGDIR,exist_ok=True)

# Account A has ~9h55m remaining. Stop launching/allowing jobs before quota edge.
DEADLINE=NB_START+9.45*3600

def manifest(slug,gname):
    ids=GROUPS[gname]
    ev=EVALROOTS[slug]
    o={}
    used=0
    for d,i,e in ids:
        gp=f"{ev}/{d}/{i}/{e}.png"
        if not os.path.exists(gp):
            continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{slug}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    p=f"{GEO}/m_{slug}_{gname}.json"
    json.dump(o,open(p,"w"))
    assert used==len(ids),(slug,gname,used,len(ids))
    return p,used

def save_all():
    for slug,d in RESULTS.items():
        jp=f"/kaggle/working/final_metric_results_{slug}.json"
        cp=f"/kaggle/working/final_metric_results_{slug}.csv"
        json.dump(d,open(jp,"w"),indent=2)
        rows=[{"model":slug,"group":g,**v} for g,v in d.items()]
        pd.DataFrame(rows).to_csv(cp,index=False)

def run_metric(gpu,slug,gname):
    man,n=manifest(slug,gname)
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_MD_SEED":"42",
        "FF_METRIC_SEED":"42",
        "TOKENIZERS_PARALLELISM":"false",
    })
    logp=f"{LOGDIR}/{slug}__{gname}.log"
    remaining=max(1,int(DEADLINE-time.time()-90))
    if remaining<600:
        return {"_timeout_before_start":True,"n":n}
    try:
        p=subprocess.run(
            [PY,"main.py","--path",man,"--use_relative_path","--base_dir",GEO,
             "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2",
             "--task","100111111","--level","0"],
            cwd=MET,env=env,capture_output=True,text=True,timeout=remaining
        )
        txt=p.stdout+p.stderr
        open(logp,"w").write(txt)
        vals={"n":n}
        for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","WRAP_E","MD"]:
            h=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
            if h:
                vals[k]=round(float(h[-1]),4)
        if p.returncode!=0:
            vals["_rc"]=p.returncode
            vals["_tail"]=txt[-2500:]
        return vals
    except subprocess.TimeoutExpired as e:
        txt=(e.stdout or "")+(e.stderr or "")
        if isinstance(txt,bytes):
            txt=txt.decode(errors="ignore")
        open(logp,"w").write(txt+"\nTIMEOUT AT ACCOUNT-A DEADLINE\n")
        return {"n":n,"_timeout":True}

# First propagate any already-completed shared groups from the partial inputs.
if complete(RESULTS["FREEFINE_BASELINE"].get("rotate_1603",{})):
    v=copy.deepcopy(RESULTS["FREEFINE_BASELINE"]["rotate_1603"])
    RESULTS["SGR_EPSREC"]["rotate_1603"]=copy.deepcopy(v)
    RESULTS["SGR_MIDHF_EPSREC"]["rotate_1603"]=copy.deepcopy(v)

if complete(RESULTS["SGR_EPSREC"].get("resize_nonsevere_1876",{})):
    RESULTS["SGR_MIDHF_EPSREC"]["resize_nonsevere_1876"]=copy.deepcopy(
        RESULTS["SGR_EPSREC"]["resize_nonsevere_1876"]
    )
elif complete(RESULTS["SGR_MIDHF_EPSREC"].get("resize_nonsevere_1876",{})):
    RESULTS["SGR_EPSREC"]["resize_nonsevere_1876"]=copy.deepcopy(
        RESULTS["SGR_MIDHF_EPSREC"]["resize_nonsevere_1876"]
    )

# Unique jobs, largest first to keep both GPUs busy.
UNIQUE=[
    # Intentionally start with unequal group sizes on GPU0/GPU1.
    ("FREEFINE_BASELINE","resize_severe_759",759,"direct"),
    ("FREEFINE_BASELINE","resize_nonsevere_1876",1876,"direct"),
    ("SGR_EPSREC","resize_severe_759",759,"direct"),
    ("SGR_EPSREC","resize_nonsevere_1876",1876,"copy_nonsev_to_midhf"),
    ("SGR_MIDHF_EPSREC","resize_severe_759",759,"direct"),
    ("FREEFINE_BASELINE","rotate_1603",1603,"copy_rotate_to_sgrs"),
]

jobs=queue.Queue()
for slug,g,n,mode in UNIQUE:
    if not complete(RESULTS[slug].get(g,{})):
        jobs.put((slug,g,n,mode))

print("unique jobs still needed:",list(jobs.queue))
lock=threading.Lock()

def apply_result(slug,g,vals,mode):
    with lock:
        if complete(vals):
            RESULTS[slug][g]=vals
            if mode=="copy_rotate_to_sgrs":
                RESULTS["SGR_EPSREC"][g]=copy.deepcopy(vals)
                RESULTS["SGR_MIDHF_EPSREC"][g]=copy.deepcopy(vals)
            elif mode=="copy_nonsev_to_midhf":
                RESULTS["SGR_MIDHF_EPSREC"][g]=copy.deepcopy(vals)
        else:
            RESULTS[slug][g]=vals
        save_all()

def worker(gpu):
    while time.time()<DEADLINE-180:
        try:
            slug,g,n,mode=jobs.get_nowait()
        except queue.Empty:
            return
        try:
            print(f"[GPU{gpu}] START {slug} / {g} (n={n})",flush=True)
            vals=run_metric(gpu,slug,g)
            apply_result(slug,g,vals,mode)
            print(f"[GPU{gpu}] DONE  {slug} / {g} -> {vals}",flush=True)
        finally:
            jobs.task_done()

threads=[threading.Thread(target=worker,args=(gpu,),daemon=True) for gpu in [0,1]]
[t.start() for t in threads]
[t.join() for t in threads]
save_all()

# Completeness audit across all three final models.
order=["all_5677","resize_2635","move_1439","rotate_1603","resize_severe_759","resize_nonsevere_1876"]
missing={}
for slug in MODELS:
    missing[slug]=[g for g in order if not complete(RESULTS[slug].get(g,{}))]
    print(slug,"MISSING",missing[slug])

all_done=all(not v for v in missing.values())
if all_done:
    print("\n✓ ALL THREE FULL FINAL METRIC TABLES COMPLETE")
else:
    print("\nPARTIAL COMBINED RESUME — save this version. The JSONs in /kaggle/working preserve completed jobs.")


unique jobs still needed: [('FREEFINE_BASELINE', 'rotate_1603', 1603, 'copy_rotate_to_sgrs')]
[GPU0] START FREEFINE_BASELINE / rotate_1603 (n=1603)
[GPU0] DONE  FREEFINE_BASELINE / rotate_1603 -> {'n': 1603, 'FID_DINO': 588.7471, 'FID_KD': 0.1423, 'FID': 47.5545, 'SUBC': 0.9008, 'BGC': 0.9673, 'WRAP_E': 0.0433, 'MD': 10.469}
FREEFINE_BASELINE MISSING []
SGR_EPSREC MISSING []
SGR_MIDHF_EPSREC MISSING []

✓ ALL THREE FULL FINAL METRIC TABLES COMPLETE


In [7]:

# ===== If complete, produce the final presentation-ready comparison immediately =====
import pandas as pd, json, os

MODELS_ORDER=["FREEFINE_BASELINE","SGR_EPSREC","SGR_MIDHF_EPSREC"]
GROUP_ORDER=["all_5677","move_1439","rotate_1603","resize_2635","resize_severe_759","resize_nonsevere_1876"]
METRICS=["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"]
ARROWS={"SUBC":"up","BGC":"up","WRAP_E":"down","MD":"down","FID":"down","FID_DINO":"down","FID_KD":"down"}

if all_done:
    rows=[]
    for g in GROUP_ORDER:
        for m in MODELS_ORDER:
            rows.append({"group":g,"model":m,**{k:RESULTS[m][g][k] for k in METRICS},
                         "n":RESULTS[m][g].get("n")})
    df=pd.DataFrame(rows)
    df.to_csv("/kaggle/working/Final_Geometry_Full5677_Results.csv",index=False)
    json.dump(RESULTS,open("/kaggle/working/Final_Geometry_Full5677_Results.json","w"),indent=2)

    delta=[]
    for g in GROUP_ORDER:
        b=RESULTS["FREEFINE_BASELINE"][g]
        for m in MODELS_ORDER[1:]:
            r=RESULTS[m][g]
            row={"group":g,"model":m}
            wins=0
            for k in METRICS:
                d=r[k]-b[k]
                row[f"{k}_delta"]=d
                if (ARROWS[k]=="up" and d>0) or (ARROWS[k]=="down" and d<0):
                    wins+=1
            row["metric_wins_vs_baseline"]=wins
            row["MD_pct_change"]=100*(r["MD"]-b["MD"])/b["MD"] if b["MD"] else None
            delta.append(row)
    ddf=pd.DataFrame(delta)
    ddf.to_csv("/kaggle/working/Final_Geometry_Full5677_Deltas.csv",index=False)

    lines=[]
    for g in GROUP_ORDER:
        lines += [f"\n## {g}\n",
                  "| Model | SUBC ↑ | BGC ↑ | WE ↓ | MD ↓ | FID ↓ | DINO ↓ | KD ↓ |",
                  "|---|---:|---:|---:|---:|---:|---:|---:|"]
        for m in MODELS_ORDER:
            v=RESULTS[m][g]
            lines.append(
                f"| {m} | {v['SUBC']:.4f} | {v['BGC']:.4f} | {v['WRAP_E']:.4f} | "
                f"{v['MD']:.4f} | {v['FID']:.4f} | {v['FID_DINO']:.4f} | {v['FID_KD']:.4f} |"
            )
    open("/kaggle/working/Final_Geometry_Full5677_Table.md","w").write("\n".join(lines))

    print("\n=== ALL 5,677 ===")
    display(df[df.group=="all_5677"][["model"]+METRICS])
    print("\n=== EXACT-AFFINE SEVERE RESIZE ===")
    display(df[df.group=="resize_severe_759"][["model"]+METRICS])
    print("\n=== DELTAS VS BASELINE ===")
    display(ddf)
    print("\n✓ FINAL GEOMETRY RESULTS READY FOR PRESENTATION")
else:
    print("Final merge skipped because at least one unique metric job is still incomplete.")



=== ALL 5,677 ===


,model,SUBC,BGC,WRAP_E,MD,FID,FID_DINO,FID_KD
0,FREEFINE_BASELINE,0.9113,0.9670,0.0474,8.5024,35.0439,487.8231,0.1433
1,SGR_EPSREC,0.9144,0.9671,0.0292,8.1779,34.7793,487.3580,0.1417
2,SGR_MIDHF_EPSREC,0.9144,0.9671,0.0292,8.1825,34.7788,487.3660,0.1417



=== EXACT-AFFINE SEVERE RESIZE ===


,model,SUBC,BGC,WRAP_E,MD,FID,FID_DINO,FID_KD
12,FREEFINE_BASELINE,0.7991,0.9651,0.0581,19.0410,60.8375,624.0966,0.1175
13,SGR_EPSREC,0.7995,0.9651,0.0581,18.6982,60.7392,624.3287,0.1152
14,SGR_MIDHF_EPSREC,0.7995,0.9651,0.0581,18.7843,60.7408,624.3306,0.1153



=== DELTAS VS BASELINE ===


,group,model,SUBC_delta,BGC_delta,WRAP_E_delta,MD_delta,FID_delta,FID_DINO_delta,FID_KD_delta,metric_wins_vs_baseline,MD_pct_change
0,all_5677,SGR_EPSREC,0.0031,0.0001,-0.0182,-0.3245,-0.2646,-0.4651,-0.0016,7,-3.816569
1,all_5677,SGR_MIDHF_EPSREC,0.0031,0.0001,-0.0182,-0.3199,-0.2651,-0.4571,-0.0016,7,-3.762467
2,move_1439,SGR_EPSREC,0.0073,0.0002,-0.0368,0.1902,-0.1217,0.8354,-0.0021,5,5.005922
3,move_1439,SGR_MIDHF_EPSREC,0.0073,0.0002,-0.0368,0.1902,-0.1217,0.8354,-0.0021,5,5.005922
4,rotate_1603,SGR_EPSREC,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.000000
5,rotate_1603,SGR_MIDHF_EPSREC,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0.000000
6,resize_2635,SGR_EPSREC,0.0026,0.0001,-0.0190,-0.4131,-0.1582,0.9387,-0.0018,6,-4.142599
7,resize_2635,SGR_MIDHF_EPSREC,0.0026,0.0001,-0.0190,-0.4002,-0.1588,0.9578,-0.0018,6,-4.013237
8,resize_severe_759,SGR_EPSREC,0.0004,0.0000,0.0000,-0.3428,-0.0983,0.2321,-0.0023,4,-1.800326
9,resize_severe_759,SGR_MIDHF_EPSREC,0.0004,0.0000,0.0000,-0.2567,-0.0967,0.2340,-0.0022,4,-1.348143



✓ FINAL GEOMETRY RESULTS READY FOR PRESENTATION
